# Profit Lab — Ideas 9–15

Inspect measured historical results separately from accounting sensitivities and invented disruption scenarios. This notebook reads reproducible results generated by `src/profit_lab.py`; it does not silently retrain or retune on June. All July 1 examples are historical, not current forecasts. The source CSV is unchanged.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown
ROOT=Path.cwd()
if not (ROOT/'reports').exists(): ROOT=ROOT.parent
p=json.loads((ROOT/'reports/profit_lab/findings.json').read_text())
m=p['modules']
print('As of:',p['asOf'],'Planning demo:',p['planningDate'])
print('Source SHA256:',p['sourceSHA256'])

As of: 2026-06-30 Planning demo: 2026-07-01
Source SHA256: f1672e728c28295b2b802d878fd38a158ba039827b429355e8438ee0463237a1


## 9. Product-level masked basket completion

Historical retrieval of an item already present in a completed basket. No item sequence or recommendation exposure exists. Compare simple popularity and conditional probability, each with probability-only or margin weighting. Losses to simple controls must remain visible.

In [2]:
display(pd.DataFrame(m['basket']['metrics']['test']).T)
print(m['basket']['metrics']['protocol'])
print(m['basket']['metrics']['interpretation'])

Jan–Apr fit, May chooses smoothing by log loss, refit Jan–May, June masked same-basket product retrieval. Alphabetically first drink anchor; order sequence absent. June previously inspected.
Matched GP is the margin of one item already present in the historical basket. It is not incremental profit or recommendation acceptance. Category AUC from earlier work does not transfer to this product-level task.


,hitAt1,hitAt3,matchedUnitGPProxy,pooledLogLoss
branch_popularity_probability,0.044571,0.105196,0.107885,0.094589
branch_popularity_profit_weighted,0.040558,0.087241,0.111762,0.094589
drink_branch_smoothed_probability,0.039290,0.108576,0.099152,0.094944
drink_branch_smoothed_profit_weighted,0.032108,0.092311,0.101127,0.094944


## 10. Smallest whole-cent policy in an explicitly defined search

Price changes hold quantity, mix, costs and discount percentages fixed. Break-even volume loss is a boundary, not a predicted response. Single-SKU candidates have an arbitrary 5% cap.

In [3]:
display(pd.DataFrame(m['dollar']['allMenuCentPolicies']))
display(pd.DataFrame(m['dollar']['singleSKUUnder5Percent']).head(10))

,branch,priceIncreaseUSD,affectedUnitsPerDay,affectedTransactionsPerDay,sameVolumeGPIncreasePerDay,recordedGPPerDay,uniformVolumeLossBreakEven,assumptions
0,Garden,0.02,98.524862,61.674033,1.959613,258.533094,0.007523,"Same quantities, mix, unit costs and discount ..."
1,Harbour,0.01,144.160221,90.243094,1.429116,540.077017,0.002639,"Same quantities, mix, unit costs and discount ..."
2,Uptown,0.02,56.138122,35.303867,1.116210,192.718564,0.005759,"Same quantities, mix, unit costs and discount ..."


,branch,product,increaseUSD,priceChangeFraction,affectedUnitsPerDay,sameVolumeGPIncreasePerDay
0,Harbour,Iced Latte,0.14,0.024911,7.425414,1.019061
1,Harbour,Butter Croissant,0.16,0.041775,6.392265,1.016221
2,Harbour,Iced Matcha,0.17,0.023644,6.193370,1.046674
3,Harbour,Classic Matcha,0.19,0.028065,5.325967,1.005950
4,Garden,Iced Latte,0.23,0.042991,4.497238,1.026867
5,Garden,Iced Matcha,0.24,0.035036,4.232044,1.010122
6,Harbour,Cold Brew,0.24,0.044362,4.270718,1.007072
7,Harbour,Cinnamon Roll,0.24,0.044362,4.292818,1.025370
8,Harbour,Strawberry Matcha,0.26,0.032379,4.005525,1.035116
9,Harbour,Vanilla Latte,0.27,0.047203,3.823204,1.025403


## 11. Substitution candidates

Same-category similarity in historical selling schedules is not customer substitution probability. Resource memberships are illustrative; confirm recipes, allergens and inventory.

In [4]:
example=next(r for r in m['substitution']['items'] if r['branch']=='Harbour' and r['product']=='Iced Latte')
display(pd.DataFrame(example['alternatives']))
print(m['substitution']['interpretation'])

Same-category candidates ranked by similarity of historical hourly sales profiles, not an estimated consumer substitution probability. Correlated schedules may share seasonality or popularity. Ingredient groups are editable serving-equivalent assumptions, not verified recipes or allergen data. No automatic transfer of unavailable-product demand.


,product,patternCosine,unitGP,price,assumedGroups
0,Iced Mocha,0.995158,4.33,6.56,"[coffee, milk]"
1,Coffee Frappe,0.992537,4.35,6.88,"[coffee, milk]"
2,Iced Americano,0.991361,3.18,4.04,[coffee]
3,Cold Brew,0.988298,4.25,5.41,[coffee]


## 12. Lower-quantile recorded-GP forecast

Train Jan–Apr, calibrate on May and freeze for June. Read coverage with quantile loss and alarm precision. No distribution-free or wartime guarantee is claimed.

In [5]:
display(pd.DataFrame(m['risk']['metrics']['June']).T)
print('Alerts:',m['risk']['metrics']['alerts'])
display(pd.DataFrame(m['risk']['nextDay']))

Alerts: {'flaggedDays': 27, 'actualLowDays': 3, 'precision': 0.1111111111111111, 'recall': 1.0}


,pinballLoss,lowerBoundCoverage,meanLowerBound
branch_train_quantile,20.685967,0.988889,204.828000
quantile_model,14.858475,0.900000,274.380051


,branch,date,lowerGP,threshold,flag
0,Garden,2026-07-01,237.554943,185.990,False
1,Harbour,2026-07-01,441.167353,382.526,False
2,Uptown,2026-07-01,138.204904,144.174,True


## 13. Discount review

Recorded discount dollars are exposure, not proven leakage. Explain unrounded-rate versus recorded-revenue rounding differences. Bigger baskets have more opportunities to contain a discounted line.

In [6]:
d=m['discount']
print('Recorded exposure:',d['totalRecordedDiscountUSD'],'Unrounded formula:',d['unroundedRateFormulaUSD'],'Rounding:',d['roundingDifferenceUSD'])
display(pd.DataFrame(d['reviewGroups']).head(12))
print(d['interpretation'])

Recorded exposure: 2178.09 Unrounded formula: 2179.3565 Rounding: 1.2664999999997235
Discount dollars measure fixed-volume financial exposure, not waste or recoverable savings. No recommendation exposure or randomized treatment exists. Larger baskets have more opportunities for a discounted line; no causal lift claim or automatic cancellation is justified.


,Branch,Category,period,lines,discountUSD,revenue,GP,margin
0,Harbour,Cold Coffee,Tue OR 19–22,319,250.98,1631.68,1094.95,0.671057
1,Harbour,Desserts,Tue OR 19–22,161,144.20,942.61,478.19,0.507304
2,Harbour,Cold Coffee,Other,187,141.79,953.82,644.23,0.675421
3,Harbour,Desserts,Other,113,105.49,686.20,348.11,0.507301
4,Harbour,Matcha & Tea,Tue OR 19–22,128,87.69,789.46,498.19,0.631052
5,Harbour,Hot Coffee,Other,148,74.30,667.51,457.09,0.684769
6,Harbour,Bakery,Tue OR 19–22,155,70.65,636.94,397.41,0.623936
7,Harbour,Matcha & Tea,Other,101,67.17,604.57,383.65,0.634583
8,Harbour,Bakery,Other,143,64.92,585.26,366.37,0.625995
9,Garden,Matcha & Tea,Tue OR 19–22,86,55.55,500.90,231.83,0.462827


## 14. Profit-aware product attention

Compare how much existing June profit three attention slots would retrieve. This is not extra profit caused by placement or promotions. Future candidate scores use forecasts and prior-period unit margins.

In [7]:
display(pd.DataFrame(m['portfolio']['metrics']).T)
print(m['portfolio']['interpretation'])

Top-three held-out retrieval of GP from products already sold in June. It measures historical profit concentration captured by a ranking, not extra GP caused by promotion. Product forecasts were selected in earlier work; June is an internal holdout. No arbitrary reassignment of demand is modeled.


,capturedHistoricalGP,captureShare
forecast_margin,4970.80,0.135168
forecast_popularity,4572.47,0.124336
historical_popularity,4720.01,0.128348


## 15. Constrained crisis scenarios

Integer stock and binary menu choices use assumed procurement, capacity and resource constraints. All stock is charged including unsold units; zero terminal salvage is assumed. No customer migration or automatic substitution. Compare all policies on 512 fresh simulated draws, separate from 16 planning draws. A lower sampled profit remains a valid negative result.

In [8]:
rows=[]
for s in m['crisis']['scenarios']:
    for policy,score in s['policies'].items(): rows.append({'scenario':s['id'],'policy':policy,**score})
display(pd.DataFrame(rows))
print('Recipe/resource assumptions:',m['crisis']['note'])
display(Markdown((ROOT/'reports/profit_lab/REPORT.md').read_text()))

Recipe/resource assumptions: Assumed one serving-equivalent of each listed resource per stocked item. Not verified recipes, supply quantities or allergens. No actual war-response dataset. Frozen budgets/capacities come from 80% of the ordinary forecast; 12 active products maximum per open branch.


,scenario,policy,expectedScenarioGP,scenarioGP10thPercentile,stockCost,preparedUnits,expectedUnsoldUnits,expectedUnservedUnits,activeMenuItems
0,ordinary,popularity_baseline,328.000117,272.9100,322.2000,165,41.390625,250.587891,36
1,ordinary,margin_baseline,327.911621,268.9360,342.0100,144,34.990234,265.187500,36
2,ordinary,optimized,337.963809,278.0490,331.1400,158,41.726562,257.923828,36
3,demand20,popularity_baseline,252.357344,203.5010,266.9000,136,37.507812,201.205078,36
4,demand20,margin_baseline,253.161914,201.9790,275.1100,119,32.072266,212.769531,36
5,demand20,optimized,254.211758,202.4340,267.2100,131,38.246094,206.943359,36
6,cost15,popularity_baseline,283.958207,233.4840,334.1440,154,35.574219,255.771484,36
7,cost15,margin_baseline,288.064719,236.1335,353.3145,143,33.449219,264.646484,36
8,cost15,optimized,291.278859,237.9350,350.3820,150,37.003906,261.201172,36
9,closed,popularity_baseline,123.931309,87.4190,177.7600,85,24.064453,129.632812,24


# Profit Lab — implemented Ideas 9–15

Seven runnable extensions accompany the root-cause investigation. They are not seven proven profit opportunities. Every module labels historical prediction, accounting sensitivity or assumed simulation. No offers, rewards, purchases, inventory transfers or menu changes are activated.

Planning examples are for **July 1, 2026**, from records through **June 30**. None is a current forecast or validated wartime model. The original transaction CSV is unchanged; the aggregate web payload contains no customer or transaction identifiers. June was inspected earlier in this project, so all new model tasks are exploratory internal holdouts, not untouched external tests.

## 9. Profit-aware basket completion

A smoothed empirical model estimates product presence given branch and an anchor drink, then ranks by probability or probability × branch/product realized unit GP. Twenty-two food candidates per beverage basket are evaluated. The alphabetically first drink is selected from the completed basket without reading food labels; the absence of item order prevents claiming real-time next-item prediction.

Fit Jan–Apr, choose smoothing from {20,100} by May pooled product-pair log loss, refit Jan–May, evaluate June. No customer history, final basket size, revenue, or purchased food labels enter prediction features. Category AUC from earlier work is not this product-level evaluation.

| Policy | June hit@1 | June hit@3 | Matched unit-GP proxy/basket | Log loss |
|---|---:|---:|---:|---:|
| branch_popularity_probability | 4.46% | 10.52% | $0.1079 | 0.09459 |
| branch_popularity_profit_weighted | 4.06% | 8.72% | $0.1118 | 0.09459 |
| drink_branch_smoothed_probability | 3.93% | 10.86% | $0.0992 | 0.09494 |
| drink_branch_smoothed_profit_weighted | 3.21% | 9.23% | $0.1011 | 0.09494 |

There are **4,734 June beverage baskets**. The conditional margin rule **loses to the branch-popularity margin rule** on hit@1 and matched GP. Profit weighting need not improve retrieval accuracy. The matched margin proxy rewards selecting an item already in the observed basket, one unit at historical GP; it is not incremental profit or observed acceptance of a recommendation.

To estimate impact: randomize no prompt, popularity prompt and margin prompt; keep eligibility and stock availability equivalent; log assignment, exposures and completed baskets; compare net total basket GP including prompt cost and purchases that would happen anyway. Do not present this model as a demonstrated improvement in sales.

## 10. The one-dollar challenge

Search a defined policy family: the smallest **whole-cent uniform price increase across a branch's menu** that adds at least $1/day at unchanged quantities, mix, unit costs and discount percentages. This is a constrained accounting search, not a universal optimum over all possible pricing/mix actions. Single-SKU candidates are also enumerated with at most a 5% list-price increase. Prices are USD as recorded; real price-step/rounding rules need verification.

| Branch | Whole-cent price step | Affected units/day | Affected transactions/day | Fixed-volume GP/day gain | Break-even uniform unit loss |
|---|---:|---:|---:|---:|---:|
| Garden | $0.02 | 98.52 | 61.67 | $1.96 | 0.752% |
| Harbour | $0.01 | 144.16 | 90.24 | $1.43 | 0.264% |
| Uptown | $0.02 | 56.14 | 35.30 | $1.12 | 0.576% |

For an assumed uniform unit loss L, daily GP change = `(1−L)×(current GP/day + fixed-volume price gain) − current GP/day`. Unit costs scale with units, with no change in product mix or fixed costs; this is not a price elasticity estimate. Tiny percentage volume losses can erase the gain. No additional purchases, discount-policy changes or real margin improvement are proven.

## 11. Menu alternatives under constraints

Generate same-category alternatives using cosine similarity of Jan–May hourly product-sales profiles. Show each alternative's recorded branch margin and the assumed resource groups. Filter candidates if a selected assumed resource is unavailable. Empty candidate sets remain empty; demand is never silently transferred.

This is a candidate-finding tool, **not a validated substitution model**. Similar hourly profiles can reflect a common calendar rather than customer switching preferences. Actual out-of-stock events, exposed alternatives, choices and recipes are needed to estimate substitution. Resource membership (coffee, milk, matcha, bakery) is an illustrative serving-equivalent proxy; it is not verified recipe or allergen information. Manual review is required before using it operationally.

## 12. Profit-at-risk forecasting

Target: recorded daily branch GP. Inputs: the eight Step 6 date/lagged-transaction features, without same-day sales or GP. A fixed gradient boosting 10th-quantile model fits Jan 8–Apr 30; May supplies a conservative one-sided calibration correction. The Jan–Apr model stays fixed for June, so calibration is not reused after a model refit. The baseline is each branch's Jan–Apr GP 10th percentile.

| June method | Pinball loss ↓ | Lower-forecast coverage | Mean lower forecast |
|---|---:|---:|---:|
| branch_train_quantile | 20.686 | 98.9% | $204.83 |
| quantile_model | 14.858 | 90.0% | $274.38 |

Lower pinball loss improves quantile forecasting relative to the conservative unconditional baseline; higher coverage alone is not the objective. The learned lower forecast has **90.0% empirical coverage** over 90 June branch-days. It is not a guaranteed financial floor. May calibration uses 93 correlated branch-days; temporal dependence, shared branches and distribution shift prevent asserting a simple distribution-free coverage guarantee.

An alert means the lower forecast crosses the branch's Jan–Apr 20th-percentile GP threshold, not a probability that intervention will help. June has **27 flags, 3 actual low days, 11.1% precision and 100.0% recall**. Many false positives and only three events make automatic interventions inappropriate. War-specific calibration or predictive performance has not been measured.

## 13. Discount review, not proven leakage

Recorded discount dollars (list value minus recorded revenue on discounted lines) total **$2,178.09**. The unrounded rate×list-value formula gives **$2,179.36**; the **$1.27** difference is line-revenue rounding reconciliation, not an additional loss. This explains small differences from earlier rate-formula summaries.

There are **153 discounted lines below a 30% recorded gross-margin rate**, with **$121.63** discount exposure. There are **0 negative-GP discounted lines**. Review tables rank branch/category/time groups by exposure. Tuesday or 19:00–22:59 defines a review period, not a causal treatment group or evidence discounts are wasted.

Exposure is a fixed-volume accounting maximum; ending promotions may change demand. Larger baskets have more chances to contain a discounted line, so their higher observed ticket does not establish discount effectiveness. Review reason codes, eligibility, campaign cost and exposure logs, then randomize a narrow policy change with a net-GP endpoint.

## 14. Product attention portfolio

Rank July 1 products using the existing product-demand forecast × Jan–May branch/product realized unit margin. Compare three top-three policies on June existing-sales GP: historical popularity, forecast popularity and forecast×margin. Each day's candidate ranking uses the previously evaluated product forecasting pipeline and prior-period economics; June is already an internal benchmark. No customer response to promotional placement is modeled.

| Top-three policy per branch/day | Captured existing June GP | Share of total June GP |
|---|---:|---:|
| forecast_margin | $4,970.80 | 13.52% |
| forecast_popularity | $4,572.47 | 12.43% |
| historical_popularity | $4,720.01 | 12.83% |

The margin ranking captures a larger share of **profit already present in historical sales**. This does not establish extra profit from moving menu items, adding promotions or replacing products. Top-k attention limits are management constraints; the UI slider changes the July 1 candidate list, while June evaluation stays fixed at three. A controlled placement test is the next evidence step.

## 15. Crisis menu/stock optimizer

For each branch/product, choose integer stock x and binary active-menu z. Across 16 assumed Poisson demand draws, maximize average `price×min(stock,demand) − scenario unit cost×all stock`. Enforce a shared procurement budget, branch unit capacities, at most 12 stocked menu products per open branch, and assumed ingredient-serving-equivalent caps. Stock must be positive for an active menu item, and unavailable/closed items cannot be stocked. All units purchased are charged, including unsold units; terminal salvage is assumed zero even for potentially storable items.

The budget is 80% of ordinary forecast procurement cost; branch capacities and shared serving-equivalent availability are 80% of ordinary forecast units. These are **invented stress-test constraints**, not observed inventory, purchasing budgets, recipes, staffing capacity or physical stock-transfer limits. Allocations are a hypothetical shared-procurement decision. No actual transfer operation is scheduled.

Six scenarios: ordinary constrained operation; demand −20%; all unit costs +15%; Harbour closure with zero customer migration; assumed milk resources unavailable; and a +20-percentage-point takeaway share that changes product mix using historical order-type profiles while preserving total expected units. None is an estimated effect of war. A closure removes Harbour's demand from the active scenario, so reported unmet units do not include the removed branch's counterfactual demand.

Popularity and margin-weighted greedy baselines obey the same scenario constraints. The MILP requests a 2% optimality gap and a 12-second limit; its status and final gap are saved. It optimizes its 16 planning draws, then all policies are evaluated on **512 separate simulated draws**, not real held-out crisis data. A feasible fallback is declared if no solver incumbent exists. No policy is guaranteed to beat a baseline on fresh samples.

| Assumed scenario | Popularity mean GP | Margin-rule mean GP | Optimized-plan mean GP | Solver gap |
|---|---:|---:|---:|---:|
| Ordinary forecast · constrained resources | $328.00 | $327.91 | $337.96 | 0.0 |
| Demand falls 20% | $252.36 | $253.16 | $254.21 | 0.0 |
| Unit costs rise 15% | $283.96 | $288.06 | $291.28 | 0.0 |
| Harbour closed · no customer migration | $123.93 | $133.82 | $134.84 | 0.0 |
| Milk serving-equivalents unavailable | $218.06 | $241.96 | $240.04 | 0.0 |
| Takeaway mix rises 20 percentage points | $323.29 | $326.82 | $333.27 | 0.0 |

**The optimized plan loses to the margin rule in the milk-unavailable scenario on fresh draws.** This result is retained. The models assume independent Poisson product demand and ignore cross-product substitution, correlated demand, traffic migration, real shelf lives, labor, rent, service/variety requirements and logistics. Narrow menus can sacrifice service; a real policy needs explicit coverage constraints. Simulated GP and its 10th percentile are conditional on this artificial world, not validated crisis savings or survival guarantees.

## Files, reproduction and checks

- `src/profit_lab.py`: Ideas 9–14, historical evaluation and scenario orchestration.
- `src/crisis_optimizer.py`: constrained menu/stock MILP and feasible baselines.
- `reports/profit_lab/findings.json`: full machine-readable evidence and assumptions.
- `drink-lens/dist/profit-lab.html`: seven interactive modules, controls, model comparisons and downloadable evidence.
- `notebooks/04_profit_lab.ipynb`: executed inspection notebook.
- `artifacts/profit_lab/`: local risk model and row-level evaluation records, ignored for publication.

```sh
python3 src/profit_lab.py --data /path/to/stories_transactions.csv --features daily_demand_features.csv
python3 src/build_profit_lab_report.py
python3 src/build_profit_lab_notebook.py
python3 src/verify_profit_lab.py --data /path/to/stories_transactions.csv
cd drink-lens
npm test
node tests/profit-lab-browser.mjs
```

The product-level modules require the existing `artifacts/demand_predictions.csv` and `artifacts/next_day_forecast.csv`; regenerate them with `src/forecast.py` and `src/predict.py` if absent. Browser tests use a temporary local server and do not send offers, modify inventory or execute payments. Source and aggregate assets are published privately; the raw CSV and row-level records remain local.

The main submission should still lead with the root-cause investigation. Breadth is not evidence of business value; negative results and missing information are part of the submission.
